# 1. Importations

## 1.1 Importations des librairies

In [ ]:
import os

In [ ]:
os.environ["KERAS_BACKEND"]="torch"
import gc
import json
import math
import time
import keras
import torch
import collections
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
from keras import layers, ops
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from keras.applications.inception_v3 import InceptionV3, preprocess_input
import tensorflow as tf

Comme pour les livrables précédents, Keras utilise PyTorch comme backend : c'est ce qui permet d'entraîner sur le GPU sous Windows.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Backend Keras : {keras.backend.backend()}")
print(f"Device d'entraînement : {DEVICE}")

## 1.2 Chemins

In [ ]:
PROJECT_DIR = os.path.abspath(os.getcwd())
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
CLASSIFIER_WEIGHTS = os.path.join(MODELS_DIR, "model_full.weights.h5")
DENOISER_MODEL = os.path.join(MODELS_DIR, "unet_denoiser_livrable2.keras")
DATASETS_DIR = os.path.join(PROJECT_DIR, "datasets", "Livrable 3")
COCO_DIR = os.path.join(DATASETS_DIR, "train2014")
PHOTOS_DIR = os.path.join(DATASETS_DIR, "photos")
DENOISED_DIR = os.path.join(DATASETS_DIR, "denoised_images")
ANNOTATIONS_DIR = os.path.join(DATASETS_DIR, "annotations")
ANNOTATION_FILE = os.path.join(ANNOTATIONS_DIR, "captions_train2014.json")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")

# 2. Prétraitement des données

## 2.1 Étape 1 : classification photo / non-photo
Le modèle du livrable 1 reconnaît cinq classes : Painting, Photo, Schematics, Sketch et Text. On garde ses cinq sorties, puis on regroupe sa prédiction en deux groupes : photo et non-photo.

Pour chaque image COCO, le code :

- la redimensionne en 128 × 128 ;
- la passe au modèle, qui divise ses pixels par 255 grâce à sa couche `Rescaling` ;
- récupère la probabilité associée à la classe Photo ;
- la compare au seuil 0,2301, défini dans le livrable 1.

Si la probabilité atteint le seuil, l'image est considérée comme une photo et elle est **déplacée** (pas copiée) dans `datasets/Livrable 3/photos`. Les autres images restent dans `train2014`.

In [ ]:
CLASS_NAMES = ["Painting", "Photo", "Schematics", "Sketch", "Text"]
NUM_CLASSES = len(CLASS_NAMES)
PHOTO_CLASS_INDEX = CLASS_NAMES.index("Photo")
THRESHOLD = 0.2301

In [ ]:
def load_classifier_model():
    model = keras.models.load_model(CLASSIFIER_WEIGHTS, compile=False)
    return model

def load_image_as_array(path, target_size=(128, 128)):
    with Image.open(path) as image:
        image = image.convert("RGB").resize(target_size)
        return np.asarray(image, dtype=np.float32)

def classify_images(image_dir: str | Path,classifier: keras.models.Model):
    results = []
    image_dir = Path(image_dir)

    for image_file in tqdm(image_dir.iterdir()):
        image_array = load_image_as_array(str(image_file))
        image_array = np.expand_dims(image_array, axis=0)

        probabilities = classifier.predict(image_array,verbose=0)
        proba_photo = float(probabilities[0][PHOTO_CLASS_INDEX])

        results.append((str(image_file),proba_photo,proba_photo >= THRESHOLD))

    return results

def move_photos_to_folder(classification_results, dest_dir=PHOTOS_DIR):
    os.makedirs(dest_dir, exist_ok=True)
    moved = []
    for path, _, is_photo in classification_results:
        if is_photo:
            if os.path.abspath(os.path.dirname(path)) != os.path.abspath(dest_dir):
                filename = os.path.basename(path)
                dest_path = os.path.join(dest_dir, filename)
                # Un nom COCO identifie une seule image : si elle existe déjà, on l'écrase.
                # Garder le nom intact permet de retrouver son image_id dans les annotations.
                os.replace(path, dest_path)
                path = dest_path
            moved.append(path)
    return moved

## 2.2 Étape 2 : débruitage avec le U-Net du livrable 2
Chaque photo retenue est chargée en RGB, redimensionnée en 299 × 299 et convertie en valeurs entre 0 et 1. Le U-Net du livrable 2 la débruite, et le résultat est enregistré dans `datasets/Livrable 3/denoised_images`.

Le U-Net divise deux fois la taille de l'image par deux (deux poolings), puis concatène les sorties de l'encodeur et du décodeur à la même résolution. La taille doit donc être un multiple de 4 : le code complète l'image de 299 à 300 pixels en recopiant les bords, puis recoupe le résultat à 299 × 299.

In [ ]:
def load_denoiser_model():
    denoiser = keras.models.load_model(DENOISER_MODEL, compile=False)
    return denoiser

def load_image_for_denoiser(path, target_size=(299, 299)):
    with Image.open(path) as image:
        image = image.convert("RGB").resize(target_size)
        return np.asarray(image, dtype=np.float32) / 255.0

def denoise_images(image_paths, denoiser, output_dir=DENOISED_DIR):
    os.makedirs(output_dir, exist_ok=True)

    for path in tqdm(image_paths):
        arr = load_image_for_denoiser(path)
        height, width = arr.shape[:2]
        pad_height = (-height) % 4
        pad_width = (-width) % 4
        padded = np.pad(arr, ((0, pad_height), (0, pad_width), (0, 0)), mode="edge")
        batch = np.expand_dims(padded, axis=0)
        denoised = keras.ops.convert_to_numpy(denoiser(batch, training=False))[0]
        denoised = denoised[:height, :width, :]
        denoised = np.clip(denoised, 0.0, 1.0)

        denoised_img = Image.fromarray(np.rint(denoised * 255).astype("uint8"))
        filename = os.path.basename(path)
        out_path = os.path.join(output_dir, filename)
        denoised_img.save(out_path)
        gc.collect()
        torch.cuda.empty_cache()


## 2.3 Étape 3 : extraction des caractéristiques avec InceptionV3

InceptionV3, pré-entraîné sur ImageNet, est utilisé sans sa tête de classification et sans pooling. Pour chaque image débruitée, sa dernière couche convolutive renvoie une grille 8 × 8 × 2048, remise en forme en (64, 2048) : 64 zones de l'image, chacune décrite par 2048 caractéristiques. Ce format permet ensuite au mécanisme d'attention de choisir les zones à regarder.

Les caractéristiques de chaque image sont enregistrées dans un fichier `<image>.jpg.npy`, à côté de l'image débruitée.

In [ ]:
def build_inception_feature_extractor():
    image_model = InceptionV3(weights="imagenet", include_top=False)
    new_input = keras.Input(shape=(299, 299, 3))
    return keras.Model(new_input, image_model(new_input))


def load_image_for_inception(path, target_size=(299, 299)):
    with Image.open(path) as image:
        image = image.convert("RGB").resize(target_size)
        image = np.asarray(image, dtype="float32")
    return preprocess_input(image), path


def extract_and_save_inception_features(img_name_vector, feature_extractor, batch_size=8):
    image_paths = sorted(set(img_name_vector))
    for start in tqdm(range(0, len(image_paths), batch_size)):
        paths = image_paths[start:start + batch_size]
        images = np.stack([load_image_for_inception(path)[0] for path in paths])
        batch_features = feature_extractor(images, training=False)
        batch_features = ops.reshape(
            batch_features,
            (batch_features.shape[0], -1, batch_features.shape[-1])
        )
        for feature, path in zip(ops.convert_to_numpy(batch_features), paths):
            np.save(path, feature)

## 2.4 Étape 4 : création du dataset

### Prétraitement des annotations
Les légendes viennent de `captions_train2014.json`. Chaque image y est identifiée par son `image_id`, qui se retrouve dans le nom du fichier : on ne garde que les images présentes dans `denoised_images`. Chaque légende est encadrée par `<start>` et `<end>`.

Le tokenizer garde les 5000 mots les plus fréquents (les autres deviennent `<unk>`), puis toutes les légendes sont complétées par des `<pad>` (indice 0) jusqu'à la longueur de la plus longue.

In [ ]:
def create_captions_and_path_vectors(annotation_file, already_done=False):
    image_path_to_caption = collections.defaultdict(list)
    train_captions = []
    img_name_vector = []
    mark_start, mark_end = '<start> ', ' <end>'
    end_path = '.npy' if already_done else ''

    with open(annotation_file, 'r', encoding='utf-8') as f:
        annotations = json.load(f)

    for val in annotations['annotations']:
        caption = mark_start + val['caption'].strip() + mark_end
        image_path = os.path.join(
            DENOISED_DIR,
            'COCO_train2014_' + '%012d.jpg' % val['image_id'] + end_path
        )
        if os.path.isfile(image_path):
            image_path_to_caption[image_path].append(caption)

    for image_path, caption_list in image_path_to_caption.items():
        train_captions.extend(caption_list)
        img_name_vector.extend([image_path] * len(caption_list))

    return img_name_vector, train_captions

def calc_max_length(tensor):
    return max(len(t) for t in tensor)

def annotation_preprocessing(train_captions, top_k=5000):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=top_k,
                                                    oov_token="<unk>",
                                                    filters='!"#$%&()*+.,-/:;=?@[\\]^_`{|}~')

    tokenizer.fit_on_texts(train_captions)
    tokenizer.word_index['<pad>'] = 0
    tokenizer.index_word[0] = '<pad>'
    train_seqs = tokenizer.texts_to_sequences(train_captions)
    cap_vector = keras.preprocessing.sequence.pad_sequences(train_seqs, padding='post')
    max_length = calc_max_length(train_seqs)

    return cap_vector, max_length, tokenizer

### Création des jeux d'entraînement et de validation
Le découpage se fait **par image** : 80 % des images pour l'entraînement, 20 % pour la validation, avec toutes leurs légendes (environ 5 par image). Une même image ne peut donc pas se retrouver dans les deux jeux. Un `DataLoader` PyTorch charge ensuite les fichiers `.npy` par batchs.

In [ ]:
class FeatureCaptionDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, captions):
        self.image_paths = image_paths
        self.captions = captions

    def __len__(self):
        return len(self.captions)

    def __getitem__(self, index):
        return np.load(self.image_paths[index]).astype("float32"), self.captions[index]


def data_splitting(img_name_vector, cap_vector, batch_size):
    img_to_cap_vector = collections.defaultdict(list)
    for img, cap in zip(img_name_vector, cap_vector):
        img_to_cap_vector[img].append(cap)

    img_keys_shuffled = shuffle(list(img_to_cap_vector.keys()), random_state=42)
    slice_index_t = int(len(img_keys_shuffled) * 0.8)
    train_keys = img_keys_shuffled[:slice_index_t]
    val_keys = img_keys_shuffled[slice_index_t:]
    train_paths = [path for path in train_keys for _ in img_to_cap_vector[path]]
    train_caps = [cap for path in train_keys for cap in img_to_cap_vector[path]]
    val_paths = [path for path in val_keys for _ in img_to_cap_vector[path]]
    val_caps = [cap for path in val_keys for cap in img_to_cap_vector[path]]

    # Les features sont stockées sous la forme image.jpg.npy dans DENOISED_DIR,
    # mais l'évaluation doit recevoir les images .jpg présentes dans PHOTOS_DIR.
    img_name_val = [
        os.path.join(PHOTOS_DIR, Path(path).stem)
        for path in val_keys
        for _ in img_to_cap_vector[path]
    ]
    cap_val = [
        cap
        for path in val_keys
        for cap in img_to_cap_vector[path]
    ]

    dataset = FeatureCaptionDataset(train_paths, train_caps)
    train_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        pin_memory=DEVICE.type == "cuda",
    )

    validation_dataset = FeatureCaptionDataset(val_paths, val_caps)
    validation_loader = torch.utils.data.DataLoader(
        validation_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        pin_memory=DEVICE.type == "cuda",
    )

    return train_loader, validation_loader, img_name_val, cap_val

## 2.5 Exécution du workflow de prétraitement

In [ ]:
def run_extraction_workflow():
    print("1. Classification photo / non-photo")
    classifier = load_classifier_model()
    classification_results = classify_images(COCO_DIR, classifier)
    photos_moved = move_photos_to_folder(classification_results)
    print(f"{len(photos_moved)} photos déplacées dans {PHOTOS_DIR}.")

    print("2. Débruitage")
    denoiser_model = load_denoiser_model()
    denoise_images(photos_moved, denoiser_model)

    print("3. Chargement des données et prétraitement")
    img_name_vector, _ = create_captions_and_path_vectors(annotation_file=ANNOTATION_FILE)

    print("4. Extraction des caractéristiques avec InceptionV3")
    feature_extractor = build_inception_feature_extractor()
    extract_and_save_inception_features(img_name_vector, feature_extractor)
    print(f"Extraction terminée dans : {DENOISED_DIR}")

# La création du DataLoader est volontairement exclue de cette cellule.

La fonction `run_extraction_workflow` enchaîne les étapes 1 à 4, mais elle n'est pas appelée ici : le workflow a déjà été exécuté en local et ses résultats (les fichiers `.npy`) sont dans `datasets/Livrable 3/denoised_images`.

## 2.6 Chargement du dataset
`create_dataset` relit les légendes des images déjà traitées (`already_done=True`), les tokenise puis crée les jeux d'entraînement et de validation.

In [ ]:
def create_dataset():
    img_name_vector, train_captions = create_captions_and_path_vectors(ANNOTATION_FILE, already_done=True)

    cap_vector, max_length, tokenizer = annotation_preprocessing(train_captions, top_k=TOP_K)

    dataset, validation_loader, img_name_val, cap_val = data_splitting(
        img_name_vector, cap_vector, BATCH_SIZE
    )

    return dataset, validation_loader, tokenizer, max_length, img_name_val, cap_val


# 3. Modèles de légendage : comparaison GRU / LSTM

Les deux modèles utilisent :

- le même `CNN_Encoder` ;
- la même attention de Bahdanau ;
- le même dataset d'entraînement et de validation ;
- les mêmes fonctions d'entraînement et d'évaluation.

Seule la cellule récurrente du décodeur change : `GRU` ou `LSTM`.

## 3.1 – Hyperparamètres communs

In [ ]:
BATCH_SIZE = 2048
EMBEDDING_DIM = 256
UNITS = 512
TOP_K = 5000            # mots gardés par le tokenizer
VOCAB_SIZE = TOP_K + 1  # + le token <pad>
EPOCHS = 20
LEARNING_RATE = 4e-3
DROPOUT = 0.3
LOG_EVERY = 100
ATTENTION_FEATURES_SHAPE = 64
L1_RATE = L2_RATE = 1e-5


## 3.2 – Encodeur et attention partagés

In [ ]:
class CNN_Encoder(keras.Model):
    def __init__(self, embedding_dim,dropout_rate, l1_rate, l2_rate):
        super().__init__()

        regularizer = keras.regularizers.L1L2(l1 = l1_rate, l2 = l2_rate)
        self.fc = layers.Dense(embedding_dim, kernel_regularizer=regularizer)

        self.dropout = layers.Dropout(dropout_rate)

    def call(self, features, training = False):
        x = ops.relu(self.fc(features))
        return self.dropout(x, training = training )


class BahdanauAttention(layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W_features = layers.Dense(units)
        self.W_hidden = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, features, hidden):
        hidden_expanded = ops.expand_dims(hidden, axis=1)
        scores = self.V(
            ops.tanh(
                self.W_features(features)
                + self.W_hidden(hidden_expanded)
            )
        )
        weights = ops.softmax(scores, axis=1)
        context = ops.sum(weights * features, axis=1)
        return context, weights

## 3.3 – Décodeur commun GRU / LSTM

In [ ]:
class AttentionDecoder(keras.Model):
    def __init__(self, embedding_dim, units, vocab_size, rnn_type, dropout_rate, l1_rate, l2_rate):
        super().__init__()
        if rnn_type not in ("gru", "lstm"):
            raise ValueError("rnn_type doit être 'gru' ou 'lstm'.")
        self.rnn_type = rnn_type
        self.units = units
        regularizer = keras.regularizers.L1L2(l1 = l1_rate, l2 = l2_rate)
        self.embedding = layers.Embedding(vocab_size, embedding_dim)
        self.attention = BahdanauAttention(units)
        if rnn_type == "gru":
            self.rnn = layers.GRU(
                units,
                return_sequences=True,
                return_state=True,
                dropout=dropout_rate,
                recurrent_initializer="glorot_uniform",
            )
        else:
            self.rnn = layers.LSTM(
                units,
                return_sequences=True,
                return_state=True,
                dropout= dropout_rate,
                recurrent_initializer="glorot_uniform",
            )
        self.fc1 = layers.Dense(units, kernel_regularizer=regularizer)
        self.dropout = layers.Dropout(dropout_rate)
        self.fc2 = layers.Dense(vocab_size, kernel_regularizer=regularizer)

    def call(self, token, features, state, training=False):
        hidden = state[0] if self.rnn_type == "lstm" else state
        context, attention_weights = self.attention(features, hidden)
        embedded = self.embedding(token)
        recurrent_input = ops.concatenate(
            [ops.expand_dims(context, axis=1), embedded], axis=-1
        )
        if self.rnn_type == "gru":
            output, next_state = self.rnn(
                recurrent_input, initial_state=state, training=training
            )
        else:
            output, next_hidden, next_cell = self.rnn(
                recurrent_input, initial_state=state, training=training
            )
            next_state = (next_hidden, next_cell)
        output = self.fc1(output)
        output = self.dropout(output, training = training)
        logits = self.fc2(output)
        logits = ops.squeeze(logits, axis=1)
        return logits, next_state, attention_weights

    def reset_state(self, batch_size):
        hidden = ops.zeros((batch_size, self.units))
        if self.rnn_type == "gru":
            return hidden
        return hidden, ops.zeros((batch_size, self.units))

## 3.4 – Dataset commun et instanciation des modèles

In [ ]:
dataset, validation_loader, tokenizer, max_length, img_name_val, cap_val = create_dataset()


In [ ]:
dummy_token = torch.zeros((1, 1), dtype=torch.int64, device=DEVICE)
dummy_features = torch.zeros((1, 64, 2048), device=DEVICE)

#gru
gru_encoder = CNN_Encoder(EMBEDDING_DIM, dropout_rate= DROPOUT, l1_rate= L1_RATE, l2_rate= L2_RATE)
gru_decoder = AttentionDecoder(EMBEDDING_DIM, UNITS,  VOCAB_SIZE, rnn_type="gru",dropout_rate= DROPOUT, l1_rate=L1_RATE, l2_rate=L2_RATE)
gru_optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
gru_encoded_features = gru_encoder(dummy_features)
gru_decoder(
    dummy_token, gru_encoded_features, gru_decoder.reset_state(1), training=False
)

#lstm
lstm_encoder = CNN_Encoder(EMBEDDING_DIM, dropout_rate= DROPOUT, l1_rate= L1_RATE, l2_rate= L2_RATE )
lstm_decoder = AttentionDecoder(EMBEDDING_DIM, UNITS, VOCAB_SIZE, rnn_type="lstm", dropout_rate= DROPOUT, l1_rate= L1_RATE, l2_rate= L2_RATE)
lstm_optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
lstm_encoded_features = lstm_encoder(dummy_features)
lstm_decoder(dummy_token, lstm_encoded_features, lstm_decoder.reset_state(1), training=False)
print('modèles instanciés')

## 3.5 – Fonctions communes d'entraînement et de validation

In [ ]:
loss_object = keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction="none"
)


def masked_loss(real, predictions):
    mask = real != 0
    losses = loss_object(real, predictions)
    return (losses * ops.cast(mask, losses.dtype)).sum(), mask.sum()


def train_step(encoder, decoder, optimizer, img_tensor, target):
    images = torch.as_tensor(img_tensor, dtype=torch.float32, device=DEVICE)
    target = torch.as_tensor(target, dtype=torch.int64, device=DEVICE)
    batch_size = target.shape[0]
    state = decoder.reset_state(batch_size)
    start_token = tokenizer.word_index["<start>"]
    decoder_input = torch.full((batch_size, 1), start_token, dtype=torch.int64, device=DEVICE)
    # training=True : sans lui, le dropout de l'encodeur ne s'active jamais
    features = encoder(images, training=True)
    total_loss = torch.zeros((), dtype=torch.float32, device=DEVICE)
    total_tokens = torch.zeros((), dtype=torch.float32, device=DEVICE)

    for index in range(1, target.shape[1]):
        predictions, state, _ = decoder(decoder_input, features, state, training=True)
        step_loss, _ = masked_loss(target[:, index], predictions)
        total_loss = total_loss + step_loss
        total_tokens = total_tokens + (target[:, index] != 0).sum()
        decoder_input = target[:, index:index + 1]

    # Pénalité elastic net : fit() l'ajoute automatiquement, pas une boucle écrite à la main.
    # total_loss est une somme sur les tokens, on multiplie donc la pénalité par leur nombre
    # pour garder le même rapport perte / pénalité qu'avec une perte moyenne.
    regularization_losses = encoder.losses + decoder.losses
    objective = total_loss
    if regularization_losses:
        objective = total_loss + sum(regularization_losses) * total_tokens

    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = torch.autograd.grad(objective, [variable.value for variable in variables], allow_unused=False)
    optimizer.apply_gradients(zip(gradients, variables))
    return (total_loss / ops.maximum(total_tokens, 1.0)).detach()


def validation_step(encoder, decoder, img_tensor, target):
    images = torch.as_tensor(img_tensor, dtype=torch.float32, device=DEVICE)
    target = torch.as_tensor(target, dtype=torch.int64, device=DEVICE)
    batch_size = target.shape[0]
    state = decoder.reset_state(batch_size)
    start_token = tokenizer.word_index["<start>"]
    decoder_input = torch.full((batch_size, 1), start_token, dtype=torch.int64, device=DEVICE)
    total_loss = torch.zeros((), dtype=torch.float32, device=DEVICE)
    total_tokens = torch.zeros((), dtype=torch.float32, device=DEVICE)
    correct_count = 0

    with torch.no_grad():
        features = encoder(images)
        for index in range(1, target.shape[1]):
            predictions, state, _ = decoder(
                decoder_input, features, state, training=False
            )
            step_loss, step_tokens = masked_loss(
                target[:, index], predictions
            )
            total_loss = total_loss + step_loss
            total_tokens = total_tokens + step_tokens
            valid_predictions = target[:, index] != 0
            correct_predictions = (
                torch.argmax(predictions, dim=-1) == target[:, index]
            )
            correct_count += int(
                (correct_predictions & valid_predictions).sum().item()
            )
            decoder_input = target[:, index:index + 1]
    return total_loss.detach(), total_tokens.detach(), correct_count

## 3.6 – BLEU et évaluation autoregressive communes

In [ ]:
def bleu_score(encoder, decoder, img_tensor, target):
    images = torch.as_tensor(img_tensor, dtype=torch.float32, device=DEVICE)
    target = torch.as_tensor(target, dtype=torch.int64, device=DEVICE)
    batch_size = target.shape[0]
    state = decoder.reset_state(batch_size)
    start_token = tokenizer.word_index["<start>"]
    end_token = tokenizer.word_index.get("<end>")
    decoder_input = torch.full((batch_size, 1), start_token, dtype=torch.int64, device=DEVICE)
    generated = [[] for _ in range(batch_size)]
    finished = [False] * batch_size

    with torch.no_grad():
        features = encoder(images)
        for _ in range(target.shape[1] - 1):
            predictions, state, _ = decoder(
                decoder_input, features, state, training=False
            )
            tokens = torch.argmax(predictions, dim=-1)
            for sample, token in enumerate(tokens.tolist()):
                # La légende d'un échantillon s'arrête à son premier <end>
                if finished[sample]:
                    continue
                if token == end_token:
                    finished[sample] = True
                else:
                    generated[sample].append(str(token))
            if all(finished):
                break
            decoder_input = tokens.unsqueeze(1)

    smoothing = SmoothingFunction().method1
    scores = []
    for sample in range(batch_size):
        reference = [
            str(int(token))
            for token in target[sample].tolist()
            if int(token) not in (0, start_token, end_token)
        ]
        candidate = generated[sample]
        scores.append(
            sentence_bleu(
                [reference],
                candidate,
                weights=(0.25, 0.25, 0.25, 0.25),
                smoothing_function=smoothing,
            ) if reference and candidate else 0.0
        )
    return float(np.mean(scores))


def feature_path_for_image(image_path):
    return os.path.join(
        DENOISED_DIR,
        Path(image_path).name + ".npy",
    )


def evaluate(encoder, decoder, image_path):
    feature_path = feature_path_for_image(image_path)
    if not os.path.isfile(feature_path):
        raise FileNotFoundError(
            f"Feature Inception introuvable pour l'image : {feature_path}"
        )
    saved_features = np.load(feature_path).astype("float32")
    features = torch.as_tensor(
        np.expand_dims(saved_features, axis=0),
        dtype=torch.float32,
        device=DEVICE,
    )
    state = decoder.reset_state(1)
    decoder_input = torch.tensor(
        [[tokenizer.word_index["<start>"]]],
        dtype=torch.int64,
        device=DEVICE,
    )
    attention_plot = np.zeros((max_length, ATTENTION_FEATURES_SHAPE))
    result = []

    with torch.no_grad():
        features = encoder(features)
        for index in range(max_length):
            predictions, state, attention = decoder(
                decoder_input, features, state, training=False
            )
            attention_plot[index] = ops.convert_to_numpy(
                ops.reshape(attention, (-1,))
            )
            token = int(torch.argmax(predictions[0]).item())
            word = tokenizer.index_word.get(token, "<unk>")
            if word == "<end>":
                break
            result.append(word)
            decoder_input = torch.tensor(
                [[token]], dtype=torch.int64, device=DEVICE
            )
    return result, attention_plot[:len(result)]

## 3.7 – Boucles d'entraînement communes

In [ ]:
def checkpoint_path(part, model_name, epoch):
    # part : "encoder" ou "decoder" ; epoch commence à 0
    return os.path.join(CHECKPOINT_DIR, f"{part}_{model_name}_epoch_{epoch:03d}.weights.h5")


def train_model(model_name, encoder, decoder, optimizer):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    train_loss_steps = []
    train_loss_history = []
    validation_steps = []
    validation_loss_history = []
    validation_accuracy_history = []
    bleu_steps = []
    bleu_history = []
    accumulated_loss = 0.0

    for epoch in range(EPOCHS):
        for batch_index, (images, targets) in enumerate(dataset):
            global_step = epoch * len(dataset) + batch_index + 1
            batch_loss = train_step(
                encoder, decoder, optimizer, images, targets
            )
            accumulated_loss += float(batch_loss.cpu())

            if global_step % LOG_EVERY == 0:
                train_loss_steps.append(global_step)
                train_loss_history.append(
                    accumulated_loss / LOG_EVERY
                )
                validation_loss, validation_accuracy = evaluate_validation(
                    encoder, decoder
                )
                validation_steps.append(global_step)
                validation_loss_history.append(validation_loss)
                validation_accuracy_history.append(validation_accuracy)
                print(
                    f"{model_name} | batch {global_step} | "
                    f"loss={train_loss_history[-1]:.4f} | "
                    f"val_loss={validation_loss:.4f} | "
                    f"accuracy={validation_accuracy:.4f}"
                )
                accumulated_loss = 0.0

        encoder.save_weights(checkpoint_path("encoder", model_name, epoch))
        decoder.save_weights(checkpoint_path("decoder", model_name, epoch))

        bleu = bleu_score(
            encoder,
            decoder,
            *next(iter(validation_loader)),
        )
        bleu_steps.append((epoch + 1) * len(dataset))
        bleu_history.append(bleu)
        print(
            f"{model_name} | époque {epoch + 1}/{EPOCHS} | "
            f"BLEU={bleu:.4f}"
        )

    return {
        "train_steps": train_loss_steps,
        "train_loss": train_loss_history,
        "validation_steps": validation_steps,
        "validation_loss": validation_loss_history,
        "validation_accuracy": validation_accuracy_history,
        "bleu_steps": bleu_steps,
        "bleu": bleu_history,
    }


def evaluate_validation(encoder, decoder):
    validation_loss = 0.0
    validation_tokens = 0.0
    validation_correct = 0
    for images, targets in validation_loader:
        batch_loss, batch_tokens, batch_correct = validation_step(
            encoder, decoder, images, targets
        )
        validation_loss += float(batch_loss.cpu())
        validation_tokens += float(batch_tokens.cpu())
        validation_correct += batch_correct
    average_loss = validation_loss / max(validation_tokens, 1.0)
    accuracy = validation_correct / max(validation_tokens, 1.0)
    return average_loss, accuracy

## 3.8 – Exécutions indépendantes

### Entraînement du GRU uniquement

In [ ]:
gru_history = train_model("GRU", gru_encoder, gru_decoder, gru_optimizer)

### Entraînement du LSTM uniquement

In [ ]:
lstm_history = train_model("LSTM", lstm_encoder, lstm_decoder, lstm_optimizer)

## 3.9 – Visualisation indépendante d'un modèle

In [ ]:
def plot_history(history, model_name):
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    axes[0].plot(
        history["train_steps"],
        history["train_loss"],
        label="Loss entraînement",
    )
    axes[0].set_title(f"{model_name} - loss tous les {LOG_EVERY} batchs")
    axes[0].set_xlabel("Batch global")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(
        history["validation_steps"],
        history["validation_loss"],
        marker="o",
        label="Validation loss",
    )
    axes[1].plot(
        history["validation_steps"],
        history["validation_accuracy"],
        marker="o",
        label="Validation accuracy",
    )
    axes[1].set_title(
        f"{model_name} - validation tous les {LOG_EVERY} batchs"
    )
    axes[1].set_xlabel("Batch global")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(
        history["bleu_steps"],
        history["bleu"],
        marker="o",
        label="BLEU en fin d'époque",
    )
    axes[2].set_title(f"{model_name} - BLEU")
    axes[2].set_xlabel("Batch global")
    axes[2].set_ylabel("BLEU")
    axes[2].set_ylim(0, 1)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_history(gru_history, "GRU")

In [ ]:
plot_history(lstm_history, "LSTM")

## 3.10 – Comparaison GRU / LSTM

Cette cellule ne lance aucun entraînement : elle superpose les historiques des deux modèles, créés dans les cellules d'entraînement de la section 3.8.

In [ ]:
def compare_histories(gru_history, lstm_history):
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    for history, name in (
        (gru_history, "GRU"),
        (lstm_history, "LSTM"),
    ):
        axes[0].plot(
            history["train_steps"],
            history["train_loss"],
            label=f"{name} - loss",
        )
        axes[1].plot(
            history["validation_steps"],
            history["validation_loss"],
            label=f"{name} - validation loss",
        )
        axes[2].plot(
            history["bleu_steps"],
            history["bleu"],
            label=f"{name} - BLEU",
        )
    axes[0].set_title("Loss d'entraînement")
    axes[1].set_title("Validation loss")
    axes[2].set_title("BLEU en fin d'époque")
    for axis in axes:
        axis.set_xlabel("Batch global")
        axis.legend()
        axis.grid(True, alpha=0.3)
    axes[2].set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

In [ ]:
compare_histories(gru_history, lstm_history)

# 4. Évaluation des modèles

## 4.1 – Score CIDEr
Le CIDEr compare chaque légende générée à **toutes** les légendes de référence de l'image, en donnant plus de poids aux mots rares et informatifs. Les légendes sont générées avec `evaluate` (section 3.6) pour chaque image de validation.

In [ ]:
%pip install pycocoevalcap

In [ ]:
from pycocoevalcap.cider.cider import Cider

def calculate_cider(encoder, decoder, img_name_val, cap_val, tokenizer):
    gts = collections.defaultdict(list)

    # 1. Préparation des références (Ground Truths)
    # On décode les séquences d'entiers en phrases textuelles pour le calcul CIDEr
    start_token = tokenizer.word_index.get('<start>')
    end_token = tokenizer.word_index.get('<end>')

    for img_path, cap_seq in zip(img_name_val, cap_val):
        words = []
        for token in cap_seq:
            # On ignore le padding, start et end
            if token not in (0, start_token, end_token):
                word = tokenizer.index_word.get(token)
                if word:
                    words.append(word)
        gts[img_path].append(" ".join(words))

    # 2. Génération des hypothèses
    res = {}
    unique_img_paths = list(set(img_name_val))

    for img_path in tqdm(unique_img_paths, desc="Génération des prédictions"):
        generated_words, _ = evaluate(encoder, decoder, img_path)
        res[img_path] = [" ".join(generated_words)]

    # 3. Calcul du score CIDEr
    scorer = Cider()
    score, _ = scorer.compute_score(gts, res)
    return score

print("Évaluation du modèle GRU avec CIDEr...")
cider_gru = calculate_cider(gru_encoder, gru_decoder, img_name_val, cap_val, tokenizer)
print(f"Score CIDEr (GRU) : {cider_gru:.4f}\n")

print("Évaluation du modèle LSTM avec CIDEr...")
cider_lstm = calculate_cider(lstm_encoder, lstm_decoder, img_name_val, cap_val, tokenizer)
print(f"Score CIDEr (LSTM) : {cider_lstm:.4f}\n")


## 4.2 – Synthèse des résultats GRU / LSTM

Les deux modèles ont été entraînés 20 époques (145 batchs de 32 légendes par époque), avec un bilan tous les 100 batchs. Ces résultats datent d'avant la correction de l'entraînement : l'elastic net et le dropout de l'encodeur n'étaient pas encore appliqués. Ils seront à mettre à jour après un nouvel entraînement.

| Mesure | GRU | LSTM |
| --- | --- | --- |
| Perte de validation minimale | 2,406 (batch 1900) | 2,391 (batch 1900) |
| Perte de validation finale | 2,427 | 2,404 |
| Perte d'entraînement finale | 2,174 | 2,151 |
| Précision sur le mot suivant (validation, finale) | 48,6 % | 48,9 % |
| BLEU en fin d'entraînement | 0,052 | 0,064 |
| CIDEr | 0,728 | 0,750 |

Le LSTM fait légèrement mieux que le GRU sur toutes les mesures, mais l'écart reste faible. Pour les deux modèles, la perte de validation ne baisse plus après le batch 1900 environ (13e époque) alors que la perte d'entraînement continue de diminuer : c'est le début du surapprentissage, avec un écart final d'environ 0,25.

Le BLEU est calculé sur un seul batch de validation et une seule légende de référence par image : il faut l'interpréter avec prudence. Le CIDEr, lui, porte sur toutes les images de validation et toutes leurs légendes.

## 4.3 – Génération de légendes avec le LSTM

On recharge le LSTM sauvegardé à la fin de la dernière époque (`checkpoints/…_LSTM_epoch_019.weights.h5` pour 20 époques), le modèle qui obtient le meilleur CIDEr (section 4.2).

Les légendes sont générées avec `evaluate` (section 3.6), par **décodage glouton** :

1. lecture des caractéristiques InceptionV3 (64 zones × 2048) déjà extraites de l'image débruitée, comme pendant l'entraînement ;
2. passage dans l'encodeur ;
3. départ de `<start>`, puis ajout à chaque pas du mot le plus probable, jusqu'à `<end>` ou `max_length` mots ;
4. conservation des poids d'attention de chaque mot pour la visualisation.

`decode_real_caption` retire `<pad>`, `<start>` et `<end>` de la légende de référence. L'affichage des images utilise les photos d'origine de `photos/` : sur une machine qui ne les a pas, seules les légendes s'affichent.

In [ ]:
# Modèle d'inférence : mêmes couches, sans dropout ni régularisation (inutiles hors entraînement)
eval_encoder = CNN_Encoder(EMBEDDING_DIM, dropout_rate=0.0, l1_rate=0.0, l2_rate=0.0)
eval_decoder = AttentionDecoder(EMBEDDING_DIM, UNITS, VOCAB_SIZE, rnn_type="lstm", dropout_rate=0.0, l1_rate=0.0, l2_rate=0.0)
eval_decoder(dummy_token, eval_encoder(dummy_features), eval_decoder.reset_state(1), training=False)

# Poids sauvegardés à la fin de la dernière époque
epoch_to_load = EPOCHS - 1
eval_encoder.load_weights(checkpoint_path("encoder", "LSTM", epoch_to_load))
eval_decoder.load_weights(checkpoint_path("decoder", "LSTM", epoch_to_load))

print(f"Modèle LSTM de l'époque {epoch_to_load + 1} chargé pour l'inférence.")

In [ ]:
def decode_real_caption(caption_tokens):
    # Légende de référence sans <pad>, <start> ni <end>
    words = []
    for token_id in caption_tokens:
        word = tokenizer.index_word.get(int(token_id), "<unk>")
        if word not in ("<pad>", "<start>", "<end>"):
            words.append(word)
    return " ".join(words)


def show_image(axis, image_path):
    # Les photos d'origine ne sont présentes que sur la machine qui a lancé le prétraitement
    if os.path.isfile(image_path):
        axis.imshow(Image.open(image_path).convert("RGB"))
        return True
    axis.text(0.5, 0.5, "Image d'origine\nnon disponible", ha="center", va="center")
    return False

## 4.4 – Visualisation de l'attention

La fonction `plot_attention` superpose les poids d'attention sur l'image pour chaque mot prédit. Cela permet de vérifier visuellement que le modèle « regarde » les bonnes régions de l'image pour chaque mot généré.

Chaque sous-figure montre :
- L'image originale en arrière-plan
- La carte d'attention (heatmap) en superposition
- Le mot prédit en titre

In [ ]:
def plot_attention(image_path, result, attention_plot):
    """
    Affiche l'image avec la carte d'attention pour chaque mot prédit.

    image_path     : chemin vers l'image originale
    result         : liste de mots prédits
    attention_plot : matrice (nb_mots, 64) des poids d'attention
    """
    if not os.path.isfile(image_path):
        print(f"Image d'origine non disponible, attention non affichée : {image_path}")
        return
    temp_image = np.array(Image.open(image_path).convert('RGB'))

    n_words = len(result)
    n_cols = min(4, n_words)
    n_rows = (n_words + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    if n_words == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for idx in range(n_words):
        # Reshape (64,) -> (8, 8) pour correspondre à la grille spatiale
        temp_att = attention_plot[idx].reshape(8, 8)

        ax = axes[idx]
        ax.set_title(result[idx], fontsize=14, fontweight='bold')
        img_disp = ax.imshow(temp_image)
        ax.imshow(
            temp_att, cmap='jet', alpha=0.5,
            extent=img_disp.get_extent(),
            interpolation='bilinear'
        )
        ax.axis('off')

    # Masquer les axes inutilisés
    for idx in range(n_words, len(axes)):
        axes[idx].axis('off')

    plt.suptitle("Visualisation de l'attention par mot", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
rid = np.random.randint(0, len(img_name_val))
image = img_name_val[rid]
real_caption = decode_real_caption(cap_val[rid])
result, attention_plot = evaluate(eval_encoder, eval_decoder, image)

print("Légende réelle  :", real_caption)
print("Légende prédite :", " ".join(result))
plot_attention(image, result, attention_plot)

## 4.5 – Légendes de plusieurs images de validation
Vingt images de validation tirées au hasard, avec la légende réelle et la légende générée par le LSTM.

In [ ]:
N_IMAGES = min(20, len(img_name_val))
N_COLS = 5
N_ROWS = math.ceil(N_IMAGES / N_COLS)

random_indices = np.random.choice(len(img_name_val), size=N_IMAGES, replace=False)

fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(25, 5 * N_ROWS))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for plot_index, data_index in enumerate(random_indices):
    image_path = img_name_val[data_index]
    real_caption = decode_real_caption(cap_val[data_index])
    predicted_words, _ = evaluate(eval_encoder, eval_decoder, image_path)
    predicted_caption = " ".join(word for word in predicted_words if word != "<unk>")

    show_image(axes[plot_index], image_path)
    axes[plot_index].axis("off")
    axes[plot_index].set_title(
        f"Réelle : {real_caption}\n"
        f"Prédite : {predicted_caption}",
        fontsize=9,
        pad=12,
        wrap=True,
    )

for index in range(N_IMAGES, len(axes)):
    axes[index].axis("off")

plt.tight_layout()
plt.show()

# 5. Pipeline complète

In [ ]:
def run_single_image_pipeline(image_path):
    import os
    import numpy as np
    from PIL import Image
    import matplotlib.pyplot as plt

    print(f"--- Lancement de la pipeline pour l'image : {os.path.basename(image_path)} ---")

    # 1. Classification Photo / Non-Photo
    print("\n[Étape 1] Classification de l'image...")
    classifier = load_classifier_model()
    image_array = load_image_as_array(image_path)
    image_batch = np.expand_dims(image_array, axis=0)
    probabilities = classifier.predict(image_batch, verbose=0)
    proba_photo = float(probabilities[0][PHOTO_CLASS_INDEX])
    is_photo = proba_photo >= THRESHOLD
    print(f"Probabilité classe 'Photo' : {proba_photo:.4f} (Seuil : {THRESHOLD})")
    print(f"Résultat de classification : {'PHOTO (Valide)' if is_photo else 'NON-PHOTO (Ignorée)'}")

    if not is_photo:
        print("L'image n'est pas classée comme une photo. Fin de la pipeline.")
        return

    # 2. Débruitage avec le U-Net
    print("\n[Étape 2] Débruitage de l'image...")
    denoiser_model = load_denoiser_model()

    # Recréation de la logique de débruitage pour une image unique
    arr = load_image_for_denoiser(image_path)
    height, width = arr.shape[:2]
    pad_height = (-height) % 4
    pad_width = (-width) % 4
    padded = np.pad(arr, ((0, pad_height), (0, pad_width), (0, 0)), mode="edge")
    batch_padded = np.expand_dims(padded, axis=0)

    denoised_batch = keras.ops.convert_to_numpy(denoiser_model(batch_padded, training=False))[0]
    denoised = denoised_batch[:height, :width, :]
    denoised = np.clip(denoised, 0.0, 1.0)

    # Sauvegarde temporaire de l'image débruitée pour InceptionV3 et l'affichage
    temp_denoised_path = os.path.join(DENOISED_DIR, os.path.basename(image_path))
    os.makedirs(DENOISED_DIR, exist_ok=True)
    denoised_img = Image.fromarray(np.rint(denoised * 255).astype("uint8"))
    denoised_img.save(temp_denoised_path)
    print(f"Image débruitée sauvegardée temporairement sous : {temp_denoised_path}")

    # 3. Extraction de features via InceptionV3
    print("\n[Étape 3] Extraction des caractéristiques de l'image...")
    feature_extractor = build_inception_feature_extractor()
    prep_image, _ = load_image_for_inception(temp_denoised_path)
    prep_image_batch = np.expand_dims(prep_image, axis=0)

    features = feature_extractor(prep_image_batch, training=False)
    features = ops.reshape(features, (features.shape[0], -1, features.shape[-1]))

    # Sauvegarde locale du fichier .npy nécessaire pour la fonction evaluate
    npy_path = temp_denoised_path + ".npy"
    np.save(temp_denoised_path, ops.convert_to_numpy(features)[0])
    print(f"Features sauvegardées sous : {npy_path}")

    # 4. Inférence (Génération de légendes) avec le LSTM réentraîné
    print("\n[Étape 4] Inférence avec le RNN (Décodeur LSTM)...")
    # Génération
    result_words, attention_plot = evaluate(eval_encoder, eval_decoder, temp_denoised_path)
    caption_generated = " ".join(result_words)
    print(f"\nLégende Générée : {caption_generated}")

    # Affichage des résultats
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(Image.open(image_path))
    axes[0].set_title("Image Originale")
    axes[0].axis("off")

    axes[1].imshow(denoised_img)
    axes[1].set_title(f"Dénoisée - Légende :\n{caption_generated}")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()